In [ ]:
from pathlib import Path
import xarray as xr
from  dask.distributed import Client
import dask

import numpy as np

### 0. Check that we're on the right machine - this is for a compute node

In [ ]:
import os
import psutil

# Check CPU Cores
logical_cores = os.cpu_count()
physical_cores = psutil.cpu_count()

# Check Memory (RAM)
virtual_mem = psutil.virtual_memory()
total_memory_gb = virtual_mem.total / (1024 ** 3)

print(f"Logical CPU Cores: {logical_cores}")
print(f"Physical CPU Cores: {physical_cores}")
print(f"Total System Memory: {total_memory_gb:.2f} GB") 

Logical CPU Cores: 96
Physical CPU Cores: 96
Total System Memory: 251.55 GB


In [ ]:
import socket

print(socket.gethostname())

n23.clstr


### 1. Settings and get a Dask client up and running

In [ ]:
resolution = 4
use_kerchunk = False

years = [str(yr) for yr in range(1959, 2023)]
months = [str(item).zfill(2) for item in range (1, 13)]
start_year = years[0]
end_year = years[-1]

filepattern = f"era5_wrf_dscale_{resolution}km"
datadir = Path(f"/beegfs/CMIP6/wrf_era5/{str(resolution).zfill(2)}km/")

if not use_kerchunk:
    dirlist = []
    for yr in years:
        dirlist.extend([p for p in datadir.joinpath(yr).glob(f"{filepattern}_*.nc")])
else:
    # Reproduce prior successful run window
    # kerchunk_catalog = Path(f"./kerchunk_refs/era5_wrf_dscale_{resolution}km_1959_2022_kerchunk.json")
    kerchunk_catalogs = [
        Path("./extraction/kerchunk_refs/era5_wrf_dscale_4km_djf_1959_2022_kerchunk.json"),
        # Path("./extraction/kerchunk_refs/era5_wrf_dscale_4km_mam_1959_2022_kerchunk.json"),
        # Path("./extraction/kerchunk_refs/era5_wrf_dscale_4km_jja_1959_2022_kerchunk.json"),
        # Path("./extraction/kerchunk_refs/era5_wrf_dscale_4km_son_1959_2022_kerchunk.json"),
    ]

spatial_chunksize = 8
dask.config.set(**{'array.slicing.split_large_chunks': True})

def attach_static_latlon(da, ds_ref):
    """Attach static XLAT/XLONG coords (Time-independent) if available."""
    out = da
    for name in ("XLAT", "XLONG"):
        if name in ds_ref.variables:
            coord = ds_ref[name]
            if "Time" in coord.dims:
                coord = coord.isel(Time=0, drop=True)
            out = out.assign_coords({name: coord})
    return out

Adapt the following to the system this is running on

In [ ]:
import os

# Point Dask spill to a large shared filesystem directory, not /tmp
_user = os.environ.get("USER", "cwaigl")
_spill_dir = f"/import/SNAP/{_user}/dask_spill"
os.makedirs(_spill_dir, exist_ok=True)

# Profile presets by workload
CLIENT_PROFILES = {
    "quantiles": {
        "n_workers": 1,
        "threads_per_worker": 14,
        "memory_limit": "110GB",
        "local_directory": _spill_dir,
        "dashboard_address": ":8787",
    },
    "rechunk": {
        "n_workers": 1,
        "threads_per_worker": 14,
        "memory_limit": "110GB",
        "local_directory": _spill_dir,
        "dashboard_address": ":8787",
    },
    "exceedance": {
        "n_workers": 4,
        "threads_per_worker": 2,
        "memory_limit": "24GB",
        "local_directory": _spill_dir,
        "dashboard_address": ":8787",
    },
}

def start_client(profile="quantiles"):
    if profile not in CLIENT_PROFILES:
        raise ValueError(f"Unknown profile '{profile}'. Choose from {list(CLIENT_PROFILES)}")

    # Ensure no old scheduler/client is still attached in this kernel
    for c in list(Client._instances):
        try:
            c.close()
        except Exception:
            pass

    cfg = CLIENT_PROFILES[profile]
    c = Client(**cfg)
    print(f"Started profile '{profile}'")
    print("Dashboard:", c.dashboard_link)
    return c

# Default profile for quantile/rechunk-heavy steps
client = start_client("quantiles")
client

Started profile 'quantiles'
Dashboard: http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 14,Total memory: 102.45 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43203,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43085,Total threads: 14
Dashboard: http://127.0.0.1:41019/status,Memory: 102.45 GiB
Nanny: tcp://127.0.0.1:36829,


2026-06-18 14:48:00,593 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:43085' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'getattr-cec82ffc-b50c-49f3-b774-3f74974a56f6', 'getattr-ed44fec0-1f3e-4f4f-8c1d-170035e503d0', 'getattr-9ddf97d1-263c-4af5-87b5-117d6eb1d963', 'open_dataset-d7645f29-6e4c-4e77-b5f4-8e50967d39dd', 'getattr-f652cd1a-512b-4d68-b13a-24655183a8f5', 'getattr-ba776618-80bb-44f6-9d64-e0c44bb14c0e', 'getattr-f87a1bc0-c77d-4608-9704-ec5ca3f5f646', 'open_dataset-f5db1ec5-d069-4a16-acd8-aa8b4dc5a298', 'open_dataset-419e6fdc-436e-4def-8dc8-c67c7f805ba9', 'open_dataset-5e69eca1-6b98-40c8-a4c8-e49b76e602ef', 'getattr-9d92aa12-2613-483e-86ab-8b9a9b79c848', 'open_dataset-d9451d43-d1be-4955-b57a-9ea0aec08334', 'open_dataset-f79eaff7-6f31-4fdd-9646-8a33c646c690', 'open_dataset-237b3909-10ba-4b5b-b314-4c76485f4264', 'open_dataset-caee45ce-bd9f-476f-8c89-e12975c972c9', 'getattr-df587de1-da7d-4c33-8c6a-eee562f3611e', 'g

### 1.1 Switch Dask profile by subtask
Use this one-liner before each major stage:
- Quantiles or rechunk-heavy operations: `client = start_client("quantiles")`
- Annual exceedance counting: `client = start_client("exceedance")`

In [ ]:
# Optional manual close at end of run
for c in list(Client._instances):
    try:
        c.close()
    except Exception:
        pass

### 2. Open the dataset, if necessary with random subsampling to reduce the Dask graph size for testing

In [ ]:
# only if testing with files directly, not kerchunk catalogs
if not use_kerchunk:
    subsample = True
    percentage = 0.1
    if subsample:

        np.random.seed(42) 
        sample_size = max(1, int(len(dirlist) * percentage)) 
        sampled_files = np.random.choice(dirlist, size=sample_size, replace=False)
    else:
        sampled_files = dirlist

In [ ]:
if use_kerchunk:
    for pth in kerchunk_catalogs:
        if not pth.exists():
            raise FileNotFoundError(f"Missing seasonal catalog: {pth}")

    dss = []
    for pth in kerchunk_catalogs:
        ds_i = xr.open_dataset(
            "reference://",
            engine="zarr",
            backend_kwargs={
                "consolidated": False,
                "storage_options": {"fo": str(pth), "remote_protocol": "file"},
            },
        )
        keep_vars = ["wspd10"] + [v for v in ("XLAT", "XLONG") if v in ds_i.variables]
        dss.append(ds_i[keep_vars])

    ds_full = xr.open_dataset(
        "reference://",
        engine="zarr",
        backend_kwargs={
            "consolidated": False,
            "storage_options": {
                "fo": str(kerchunk_catalog),
                "remote_protocol": "file"
            }
        }
    )
else:
    ds_full = xr.open_mfdataset(
        sampled_files,
        parallel=True,
        chunks={"Time": 8760},
        data_vars="minimal",
        coords="minimal",
        compat="override"
    )

keep_vars = ["wspd10"] + [v for v in ("XLAT", "XLONG") if v in ds_full.variables]
ds = ds_full[keep_vars].chunk({"Time": 8760})

2026-06-18 14:48:00,582 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('open_dataset-51050499-6b61-4244-b3b9-5aaf09ee4de6')" coro=<Worker.execute() done, defined at /import/SNAP/cwaigl/dyndowntools/.pixi/envs/default/lib/python3.12/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2026-06-18 14:48:00,582 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('open_dataset-419a2561-c15e-4b3a-a267-6e4712b9c3e8')" coro=<Worker.execute() done, defined at /import/SNAP/cwaigl/dyndowntools/.pixi/envs/default/lib/python3.12/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2026-06-18 14:48:00,582 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('open_dataset-0813ce09-827b-496d-9308-da22e91a17b0')" coro=<Worker.execute() done, defined at /import/SNAP/cwaigl/dyndowntools/.pixi/envs/defau

KeyboardInterrupt: 

In [ ]:
ds.Time.groupby(ds.Time.dt.year).count()

<xarray.DataArray 'Time' (year: 64)> Size: 512B
array([144,  96, 144,  96, 120, 144, 120, 168,  24,  96,  48, 120,  48,
        72,  96,  96,  48,  96,  48,  72,  72, 144,  48, 120,  24,  48,
       144,  24,  72, 120, 120,  96, 192, 120,  96, 144,  24,  48, 120,
        48,  24,  96,  96,  48, 120,  72,  48, 120,  24,  72, 144,  96,
       120, 120,  48,  96,  72, 144,  24,  48,  24,  48,  96,  72])
Coordinates:
  * year     (year) int64 512B 1959 1960 1961 1962 1963 ... 2019 2020 2021 2022

In [ ]:
# Task 1 replay: lock spatial chunking to the known successful 8x8 setting
task1_spatial_chunksize = 8
ds_for_calc = ds.chunk({"Time": -1, "south_north": task1_spatial_chunksize, "west_east": task1_spatial_chunksize})
ds_for_calc

<xarray.Dataset> Size: 558GB
Dimensions:       (Time: 5592, south_north: 450, west_east: 420,
                   interp_level: 9, soil_layers_stag: 4)
Coordinates:
  * Time          (Time) datetime64[ns] 45kB 1959-01-05 ... 2022-10-08T23:00:00
    XLONG         (south_north, west_east) float32 756kB dask.array<chunksize=(8, 8), meta=np.ndarray>
    XLAT          (south_north, west_east) float32 756kB dask.array<chunksize=(8, 8), meta=np.ndarray>
    XTIME         (Time) float32 22kB dask.array<chunksize=(5592,), meta=np.ndarray>
  * interp_level  (interp_level) float64 72B 200.0 300.0 500.0 ... 950.0 1e+03
Dimensions without coordinates: south_north, west_east, soil_layers_stag
Data variables: (12/43)
    slp           (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    ctt           (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    dbz           (Time, interp_level, south_north, west_east) float32 38GB dask.array<chunksize=(5592, 3, 8, 8), meta=np.ndarray>
    rh2           (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    T2            (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    Q2            (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    ...            ...
    u             (Time, interp_level, south_north, west_east) float32 38GB dask.array<chunksize=(5592, 3, 8, 8), meta=np.ndarray>
    v             (Time, interp_level, south_north, west_east) float32 38GB dask.array<chunksize=(5592, 3, 8, 8), meta=np.ndarray>
    w             (Time, interp_level, south_north, west_east) float32 38GB dask.array<chunksize=(5592, 3, 8, 8), meta=np.ndarray>
    rainnc        (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    rainc         (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
    acsnow        (Time, south_north, west_east) float32 4GB dask.array<chunksize=(5592, 8, 8), meta=np.ndarray>
Attributes:
    date:     2024-04-24T20:58:14.777387
    data:     Downscaled ERA5 using WRF
    info:     Alaska Climate Adaptation Science Center, University of Alaska ...
    contact:  cwaigl@alaska.edu
    version:  WRF V4.5.1 - project v. 1.1

In [ ]:
timeslist = ds_for_counts.Time
timeslist

<xarray.DataArray 'Time' (Time: 5592)> Size: 45kB
array(['1959-01-05T00:00:00.000000000', '1959-01-05T01:00:00.000000000',
       '1959-01-05T02:00:00.000000000', ..., '2022-10-08T21:00:00.000000000',
       '2022-10-08T22:00:00.000000000', '2022-10-08T23:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * Time     (Time) datetime64[ns] 45kB 1959-01-05 ... 2022-10-08T23:00:00
    XTIME    (Time) float32 22kB dask.array<chunksize=(5592,), meta=np.ndarray>

### 3.1 Calculate the 90th and 95th percentiles across the entire time dimension (lazy Dask operation)

In [ ]:
# Compute sequentially and materialize each quantile to avoid a very large combined write graph
p90 = ds_for_calc["wspd10"].quantile(0.90, dim='Time').compute()
p95 = ds_for_calc["wspd10"].quantile(0.95, dim='Time').compute()

full_percentiles = xr.concat([p90, p95], dim='percentile').assign_coords(percentile=[0.90, 0.95])
full_percentiles.name = 'wspd10'
full_percentiles = attach_static_latlon(full_percentiles, ds)
full_percentiles

<xarray.DataArray 'wspd10' (percentile: 2, south_north: 450, west_east: 420)> Size: 3MB
array([[[13.06055565, 12.99232979, 12.86858311, ..., 13.13540497,
         13.10910816, 13.13352566],
        [13.08638783, 13.0475873 , 12.98261213, ..., 13.14530525,
         13.1770689 , 13.19031811],
        [13.11435881, 13.0844861 , 13.09440575, ..., 13.2134944 ,
         13.24551821, 13.18765144],
        ...,
        [10.49200726, 10.52234821, 10.57614107, ...,  8.81745176,
          8.69798393,  8.57945299],
        [10.47815084, 10.51183195, 10.54854059, ...,  8.7775382 ,
          8.68550568,  8.54594526],
        [10.47735548, 10.503512  , 10.51639414, ...,  8.83577156,
          8.72404871,  8.59651384]],

       [[14.5726686 , 14.46811519, 14.38836827, ..., 15.04326487,
         15.07374182, 15.0946888 ],
        [14.61442895, 14.57384176, 14.49937487, ..., 15.15186915,
         15.17019186, 15.11313624],
        [14.58735213, 14.5481688 , 14.49991617, ..., 15.11075363,
         15.19334183, 15.12086716],
        ...,
        [11.9139123 , 11.98727431, 11.99945126, ..., 10.01924582,
          9.85538936,  9.74823713],
        [11.84031663, 11.88987999, 11.92218318, ...,  9.99495716,
          9.82091713,  9.64251986],
        [11.8089819 , 11.83119383, 11.93594122, ...,  9.9973403 ,
          9.8636519 ,  9.61545091]]])
Coordinates:
    XLONG       (south_north, west_east) float32 756kB -164.9 -164.9 ... -128.6
    XLAT        (south_north, west_east) float32 756kB 55.13 55.14 ... 70.52
    quantile    (percentile) float64 16B 0.9 0.95
  * percentile  (percentile) float64 16B 0.9 0.95
Dimensions without coordinates: south_north, west_east

In [ ]:
len(sampled_files)

233

In [ ]:
infix = f"{start_year}_{end_year}"
if subsample: infix += f"_subsample{percentage*100:.0f}"
fn = f'final_hourly_percentiles_90_95_{infix}.nc'

full_percentiles.to_netcdf(fn)

### 3.2 calculate annual exeedance counts for whole year at each quantile threshold

In [ ]:
fn

'final_hourly_percentiles_90_95_1959_2022_subsample1.nc'

In [ ]:
# Load thresholds lazily with chunks so comparisons stay distributed.
thresh = xr.open_dataset(
    fn,
    chunks={"percentile": 1, "south_north": spatial_chunksize, "west_east": spatial_chunksize},
)

# For exceedance counts, keep Time chunked by year-sized blocks instead of Time=-1.
ds_for_counts = ds["wspd10"].chunk(
    {"Time": 8760, "south_north": spatial_chunksize, "west_east": spatial_chunksize}
)

q_values = thresh["percentile"].values
years = np.unique(ds_for_counts.Time.dt.year.values)

annual_exceedance_list = []
for q in q_values:
    thresh_q = thresh["wspd10"].sel(percentile=q)
    yearly_counts = []

    for year in years:
        ds_year = ds_for_counts.sel(Time=str(int(year)))
        exceedance_hours = (ds_year > thresh_q).sum(dim="Time")

        annual_q = exceedance_hours.expand_dims(year=[int(year)])
        yearly_counts.append(annual_q)

    annual_q_all_years = xr.concat(yearly_counts, dim="year")
    annual_q_all_years = annual_q_all_years.expand_dims(percentile=[q])
    annual_exceedance_list.append(annual_q_all_years)

annual_exceedances = xr.concat(annual_exceedance_list, dim="percentile")
annual_exceedances = attach_static_latlon(annual_exceedances, ds)
annual_exceedances.to_netcdf(f"annual_exceedance_counts_90_95{infix}.nc")

HDF5-DIAG: Error detected in HDF5 (2.1.0) thread 2:
  #000: /home/conda/feedstock_root/build_artifacts/hdf5_1775243213841/work/src/H5F.c line 490 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: /home/conda/feedstock_root/build_artifacts/hdf5_1775243213841/work/src/H5VLcallback.c line 4109 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: /home/conda/feedstock_root/build_artifacts/hdf5_1775243213841/work/src/H5VLcallback.c line 4045 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: /home/conda/feedstock_root/build_artifacts/hdf5_1775243213841/work/src/H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: /home/conda/feedstock_root/build_artifacts/hdf5_17752432138

### 3.3 calculate seasonal percentiles (DJF, MAM, JJA, SON)

In [ ]:
import gc

seasons = ["DJF", "MAM", "JJA", "SON"]
seasonal_percentiles = []

for season in seasons:
    kerchunk_catalog = Path(
        f"./extraction/kerchunk_refs/{filepattern}_{season.lower()}_{start_year}_{end_year}_kerchunk.json"
    )
    if not kerchunk_catalog.exists():
        raise FileNotFoundError(f"Missing seasonal catalog: {kerchunk_catalog}")

    ds_season = xr.open_dataset(
        "reference://",
        engine="zarr",
        backend_kwargs={
            "consolidated": False,
            "storage_options": {
                "fo": str(kerchunk_catalog),
                "remote_protocol": "file",
            },
        },
    )[["wspd10"]]

    # Drop non-essential coords to avoid expensive coordinate shuffles.
    ds_season = ds_season.reset_coords(drop=True)

    da_season = ds_season["wspd10"].chunk(
        {"Time": -1, "south_north": spatial_chunksize, "west_east": spatial_chunksize}
    )

    q_season = da_season.quantile([0.90, 0.95], dim="Time").compute()
    q_season = q_season.assign_coords(season=season).expand_dims("season")
    seasonal_percentiles.append(q_season)

    ds_season.close()
    del da_season, q_season

    gc.collect()

ds_seasonal_per = xr.concat(seasonal_percentiles, dim="season")
ds_seasonal_per = attach_static_latlon(ds_seasonal_per, ds)
ds_seasonal_per.to_netcdf("seasonal_hourly_percentiles_90_95.nc")

In [ ]:
# import gc

# seasons = ["DJF", "MAM", "JJA", "SON"]
# seasonal_percentiles = []

# for season in seasons:
#     # Lazily filter for just the current season's months
#     ds_season = ds_for_calc.sel(Time=ds_for_calc.Time.dt.season == season)
    
#     # Compute percentiles and trigger immediately so we don't accumulate the whole
#     # time axis for all seasons in the Dask graph at once
#     per_season = (
#         ds_season["wspd10"]
#         .quantile([0.90, 0.95], dim="Time")
#         .rename({"quantile": "percentile"})
#         .assign_coords(season=season)
#         .compute()   # materialise before moving to next season
#     )
#     seasonal_percentiles.append(per_season)
#     del ds_season
#     gc.collect()

# # Combine the individual (already-computed) seasons
# ds_seasonal_per = xr.concat(seasonal_percentiles, dim="season")

# # Save the result
# ds_seasonal_per.to_netcdf("seasonal_percentiles_90_95.nc")

### 3.4 Calculate annual excedance counts

In [ ]:
# Load thresholds lazily with chunks so comparisons stay distributed.
thresh = xr.open_dataset(
    "seasonal_hourly_percentiles_90_95.nc",
    chunks={"percentile": 1, "season": 1, "south_north": spatial_chunksize, "west_east": spatial_chunksize},
)

# For exceedance counts, keep Time chunked by year-sized blocks instead of Time=-1.
ds_for_counts = ds["wspd10"].chunk(
    {"Time": 8760, "south_north": spatial_chunksize, "west_east": spatial_chunksize}
)

q_values = thresh["percentile"].values
years = np.unique(ds_for_counts.Time.dt.year.values)

annual_exceedance_list = []
for q in q_values:
    thresh_q = thresh["wspd10"].sel(percentile=q)
    yearly_counts = []

    for year in years:
        ds_year = ds_for_counts.sel(Time=str(int(year)))
        seasonal_counts = []

        for season in thresh_q.season.values:
            ds_year_season = ds_year.sel(Time=ds_year.Time.dt.season == season)
            thresh_season = thresh_q.sel(season=season)

            exceedance_hours = (ds_year_season > thresh_season).sum(dim="Time")
            seasonal_counts.append(exceedance_hours)

        annual_q = xr.concat(seasonal_counts, dim="season").sum(dim="season")
        annual_q = annual_q.expand_dims(year=[int(year)])
        yearly_counts.append(annual_q)

    annual_q_all_years = xr.concat(yearly_counts, dim="year")
    annual_q_all_years = annual_q_all_years.expand_dims(percentile=[q])
    annual_exceedance_list.append(annual_q_all_years)

annual_exceedances = xr.concat(annual_exceedance_list, dim="percentile")
annual_exceedances = attach_static_latlon(annual_exceedances, ds)
annual_exceedances.to_netcdf("annual_exceedance_counts_90_95_byseason.nc")

KeyError: "No variable named 'percentile'. Variables on the dataset include ['XLONG', 'XLAT', 'quantile', 'season', 'wspd10']"

### Cleanup

In [ ]:
import gc

# clean up any prior objects in this kernel
closed = 0
for c in list(Client._instances):
    try:
        c.close()
        closed += 1
    except Exception as e:
        print(f"Close warning: {e}")

# 2) Close cluster handle too (if you created one explicitly)
try:
    cluster.close()
except Exception:
    pass
gc.collect()

803